Route detection from GPS data

In [ ]:
from pathlib import Path
import pandas as pd
from math import radians, sin, cos, sqrt, atan2

# =========================
# PROJECT PATHS
# =========================

PROJECT_ROOT = Path.cwd().parents[1]
DATA_FEB_DIR = PROJECT_ROOT / "Dataset_feb"
SAMSUNG_DIR = DATA_FEB_DIR / "Dataset_samsung"

INPUT_FILE = SAMSUNG_DIR / "Geo_file_Cleaned.csv"
WEATHER_FILE = DATA_FEB_DIR / "Weather_file.csv"
OUTPUT_FILE = SAMSUNG_DIR / "Routes.xlsx"

print("GPS file found:", INPUT_FILE.exists())
print("Weather file found:", WEATHER_FILE.exists())


# =========================
# ROUTE DETECTION PARAMETERS
# =========================

STOP_RADIUS_METERS = 200
STOP_TIME_MINUTES = 20

TIME_COL = "timestamp"
USER_COL = "user_id"


# =========================
# HELPER FUNCTIONS
# =========================

def haversine(lat1, lon1, lat2, lon2):
    """Calculate the distance between two GPS points in meters."""

    R = 6371000

    lat1, lon1, lat2, lon2 = map(
        radians,
        [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        sin(dlat / 2) ** 2
        + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    )

    c = 2 * atan2(
        sqrt(a),
        sqrt(1 - a)
    )

    return R * c


def get_space_columns(df):
    """Identify columns representing spatial context."""

    exclude = {
        "user_id",
        "timestamp",
        "latitude",
        "longitude",
        "altitude",
        "speed",
        "error",
        "day"
    }

    return [
        col for col in df.columns
        if col not in exclude
    ]


def get_weather_stats(start, end, weather_df):
    """Calculate weather statistics for a specific route time window."""

    if weather_df.empty:
        return None, None, None, None

    mask = (
        (weather_df["timestamp"] >= start)
        & (weather_df["timestamp"] <= end)
    )

    subset = weather_df.loc[mask]

    if subset.empty:
        return None, None, None, None

    return (
        subset["temperature_2m"].mean(),
        subset["temperature_2m"].std(),
        subset["precipitation"].mean(),
        subset["precipitation"].std()
    )


def detect_stop_segment(
    day_df,
    start_idx,
    radius_m=200,
    stop_minutes=20
):
    """Detect a stop based on spatial radius and minimum duration."""

    n = len(day_df)

    ref_lat = day_df.iloc[start_idx]["latitude"]
    ref_lon = day_df.iloc[start_idx]["longitude"]
    ref_time = day_df.iloc[start_idx][TIME_COL]

    j = start_idx + 1

    while j < n:
        lat_j = day_df.iloc[j]["latitude"]
        lon_j = day_df.iloc[j]["longitude"]

        dist = haversine(
            ref_lat,
            ref_lon,
            lat_j,
            lon_j
        )

        if dist > radius_m:
            break

        duration_minutes = (
            day_df.iloc[j][TIME_COL] - ref_time
        ).total_seconds() / 60.0

        if duration_minutes >= stop_minutes:

            k = j + 1

            while k < n:
                lat_k = day_df.iloc[k]["latitude"]
                lon_k = day_df.iloc[k]["longitude"]

                if haversine(
                    ref_lat,
                    ref_lon,
                    lat_k,
                    lon_k
                ) > radius_m:
                    break

                k += 1

            return start_idx, k

        j += 1

    return None


def extract_routes_from_day(
    day_df,
    space_cols,
    route_id_start=1
):
    """Extract mobility routes for one participant on one day."""

    routes = []
    n = len(day_df)

    if n == 0:
        return routes, route_id_start

    current_start = 0
    route_id = route_id_start

    def clean_space_string(row, cols):
        found = [
            col for col in cols
            if pd.notna(row[col])
            and str(row[col]).strip().lower() == "yes"
        ]

        if len(found) > 1:
            found = [
                value for value in found
                if value.lower() != "undefined"
            ]

        return "; ".join(found) if found else "Undefined"

    i = 0

    while i < n:

        stop_info = detect_stop_segment(
            day_df,
            i,
            STOP_RADIUS_METERS,
            STOP_TIME_MINUTES
        )

        if stop_info is None:
            i += 1
            continue

        stop_start_idx, first_idx_after_stop = stop_info
        route_end_idx = stop_start_idx - 1

        if route_end_idx >= current_start:

            route_df = day_df.iloc[
                current_start:route_end_idx + 1
            ]

            start_row = route_df.iloc[0]
            end_row = route_df.iloc[-1]

            routes.append({
                "route_id": route_id,
                "user_id": start_row[USER_COL],
                "day": start_row[TIME_COL].date(),
                "start_time": start_row[TIME_COL],
                "end_time": end_row[TIME_COL],
                "start_altitude": start_row["altitude"],
                "end_altitude": end_row["altitude"],
                "avg_speed": route_df["speed"].mean(),
                "avg_error": route_df["error"].mean(),
                "start_space_type": clean_space_string(
                    start_row,
                    space_cols
                ),
                "end_space_type": clean_space_string(
                    end_row,
                    space_cols
                )
            })

            route_id += 1

        current_start = first_idx_after_stop
        i = first_idx_after_stop

    # Include the remaining segment after the final detected stop
    if current_start < n:

        route_df = day_df.iloc[current_start:n]

        if not route_df.empty:

            start_row = route_df.iloc[0]
            end_row = route_df.iloc[-1]

            routes.append({
                "route_id": route_id,
                "user_id": start_row[USER_COL],
                "day": start_row[TIME_COL].date(),
                "start_time": start_row[TIME_COL],
                "end_time": end_row[TIME_COL],
                "start_altitude": start_row["altitude"],
                "end_altitude": end_row["altitude"],
                "avg_speed": route_df["speed"].mean(),
                "avg_error": route_df["error"].mean(),
                "start_space_type": clean_space_string(
                    start_row,
                    space_cols
                ),
                "end_space_type": clean_space_string(
                    end_row,
                    space_cols
                )
            })

            route_id += 1

    return routes, route_id


# =========================
# MAIN PIPELINE
# =========================

def main():

    # Check that the cleaned GPS dataset is available
    if not INPUT_FILE.exists():
        print(f"[ERROR] Input file not found: {INPUT_FILE}")
        return

    # =========================
    # LOAD GPS DATA
    # =========================

    df = pd.read_csv(INPUT_FILE)

    df[TIME_COL] = pd.to_datetime(
        df[TIME_COL],
        errors="coerce"
    )

    df = df.dropna(
        subset=[
            USER_COL,
            TIME_COL,
            "latitude",
            "longitude"
        ]
    )

    df = (
        df.sort_values([USER_COL, TIME_COL])
        .reset_index(drop=True)
    )

    df["day"] = df[TIME_COL].dt.date


    # =========================
    # LOAD WEATHER DATA
    # =========================

    if WEATHER_FILE.exists():

        weather_df = pd.read_csv(WEATHER_FILE)

        weather_df["timestamp"] = pd.to_datetime(
            weather_df["timestamp"]
        )

        print(f"Weather file loaded: {WEATHER_FILE}")

    else:

        print(
            f"[WARNING] Weather file not found: "
            f"{WEATHER_FILE}"
        )

        weather_df = pd.DataFrame()


    # =========================
    # DETECT ROUTES
    # =========================

    space_cols = get_space_columns(df)

    all_routes = []
    next_route_id = 1

    # Process GPS data separately for each participant and day
    for user_id, user_df in df.groupby(USER_COL):

        user_df = (
            user_df.sort_values(TIME_COL)
            .reset_index(drop=True)
        )

        for day, day_df in user_df.groupby("day"):

            routes, next_route_id = extract_routes_from_day(
                day_df,
                space_cols,
                next_route_id
            )

            all_routes.extend(routes)


    routes_df = pd.DataFrame(all_routes)


    # =========================
    # ROUTE POST-PROCESSING
    # =========================

    if not routes_df.empty:

        # Remove routes with zero duration
        routes_df = routes_df[
            routes_df["start_time"]
            != routes_df["end_time"]
        ].reset_index(drop=True)


        # =========================
        # ADD WEATHER FEATURES
        # =========================

        if not weather_df.empty:

            print(
                "Calculating weather statistics "
                "for each route..."
            )

            weather_results = routes_df.apply(
                lambda row: get_weather_stats(
                    row["start_time"],
                    row["end_time"],
                    weather_df
                ),
                axis=1,
                result_type="expand"
            )

            routes_df[
                [
                    "avg_temp",
                    "std_temp",
                    "avg_precip",
                    "std_precip"
                ]
            ] = weather_results


        # =========================
        # CLEAN SPATIAL CONTEXT LABELS
        # =========================

        for col in [
            "start_space_type",
            "end_space_type"
        ]:

            routes_df[col] = (
                routes_df[col]
                .fillna("Undefined")
                .astype(str)
                .str.strip()
            )

            routes_df.loc[
                routes_df[col].isin(
                    [
                        "",
                        "None",
                        "nan",
                        "NaN",
                        "<NA>"
                    ]
                ),
                col
            ] = "Undefined"


        # =========================
        # SAVE ROUTES
        # =========================

        routes_df.to_excel(
            OUTPUT_FILE,
            index=False
        )

        print("Route detection completed successfully!")
        print(f"Routes file created: {OUTPUT_FILE}")
        print(f"Total valid routes: {len(routes_df)}")

    else:
        print(
            "No routes were detected "
            "with the defined criteria."
        )


if __name__ == "__main__":
    main()